In [0]:
%run ./config

In [0]:

dbutils.widgets.text("batch_control_key", "", "Batch Control Key")
BATCH_CONTROL_KEY = dbutils.widgets.get("batch_control_key").strip()
assert BATCH_CONTROL_KEY, "batch_control_key parameter is required"

print(f"  Batch Control Key: {BATCH_CONTROL_KEY}")



In [0]:
# ===================================================================== #
# Read active config for this batch_control_key from control table      #
# ===================================================================== #
config_df = (
    spark.read.table(STREAMING_METADATA_TABLE)
    .filter(
        (F.col("is_active") == True) &
        (F.col("batch_control_key") == BATCH_CONTROL_KEY)
    )
)

config_rows = config_df.collect()
assert len(config_rows) > 0, (
    f"No active config found for '{BATCH_CONTROL_KEY}' in {STREAMING_METADATA_TABLE}"
)

cfg = config_rows[0].asDict()

print(f"  Source System      : {cfg['source_system']}")
print(f"  Event Hub          : {cfg['event_hub_namespace']}/{cfg['event_hub_name']}")
print(f"  Consumer Group     : {cfg.get('consumer_group', '$Default')}")
print(f"  Bronze Table       : {cfg['bronze_table_name']}")
print(f"  Checkpoint         : {cfg['checkpoint_location']}")
print(f"  Trigger Interval   : {cfg['trigger_interval']}")
print(f"  Max Events/Trigger : {cfg['max_events_per_trigger']}")



In [0]:
df=spark.read.table(f"{catalog_name}.f_control.streaming_ingestion_metadata").filter(col("batch_control_key")==lit("streaming_sales"))
df.show()
print(df.collect())
print(df.collect()[0])
print(df.collect()[0]["source_system"])
cfg=df.collect()[0].asDict()
print(cfg)
print(cfg["batch_control_key"])

In [0]:
raw_stream=(spark.readStream.format("kafka")
        .option("kafka.bootstrap.servers",BOOTSTRAP_SERVERS)
        .option("subscribe",TOPIC_NAME)
        .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", jaas_config)
        .option("startingOffsets","earliest")
        .load()
)


In [0]:

bronze_stream = (
    raw_stream
    .withColumns({
        "raw_xml_payload":          F.col("value").cast("string"),
        "_eventhub_enqueue_ts":     F.col("timestamp"),
        "_eventhub_offset":         F.col("offset").cast("int"),
        "_eventhub_partition_id":  F.col("partition").cast("int"),
        "_source_system_code":      F.lit(cfg.get("source_system_code", cfg["source_system"])),
        "_division_code":           F.lit(cfg.get("division_code", "")),
        "_data_domain":             F.lit(cfg.get("data_domain", "")),
        "_pipeline_name":           F.lit(BATCH_CONTROL_KEY),
        "_pipeline_run_id":         F.lit("abc"),
        "_created_ts":              F.current_timestamp(),
    })
)

bronze_stream = (
    bronze_stream
    .withColumns({
        "_record_guid": F.md5(F.concat_ws(
            "-",
            F.coalesce(F.col("_eventhub_partition_id").cast("string"), F.lit("")),
            F.coalesce(F.col("_eventhub_offset").cast("string"), F.lit(""))
        )),
    })
    .select(
        "_record_guid", "raw_xml_payload",
        "_eventhub_enqueue_ts", "_eventhub_offset", "_eventhub_partition_id",
        "_source_system_code", "_division_code", "_data_domain",
        "_pipeline_name", "_pipeline_run_id", "_created_ts"
    )
)

partition_col = "_created_ts"
bronze_stream = bronze_stream.withColumn(
    "partition_date",
    F.to_date(F.col(partition_col))
)

writer = (
    bronze_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", cfg["checkpoint_location"])
    .option("mergeSchema", "true")
    .option("spark.databricks.delta.optimizeWrite.enabled", "true")
    .trigger(availableNow=True)
    .partitionBy("partition_date")
)

streaming_query = writer.toTable(cfg["bronze_table_name"])

print(f"\n\u2713 Streaming query started")
print(f"  Query ID : {streaming_query.id}")
print(f"  Target   : {cfg['bronze_table_name']}")
print(f"  Partition: {partition_col}")

In [0]:



# # ===================================================================== #
# # Build Kafka options and start the streaming query                     #
# # ===================================================================== #

# starting_offset = cfg.get("starting_offset") or "-1"
# kafka_offsets = (
#     "earliest" if starting_offset == "-1"
#     else "latest" if starting_offset.lower() == "@latest"
#     else starting_offset
# )

# consumer_group = (
#     cfg.get("consumer_group")
#     or f"{BATCH_CONTROL_KEY.lower().replace('_', '-')}-stream"
# )

# kafka_opts = build_eh_kafka_options(
#     event_hub_name=cfg["event_hub_name"],
#     consumer_group=consumer_group,
#     starting_offsets=kafka_offsets,
#     max_offsets_per_trigger=cfg["max_events_per_trigger"],
# )

# print(f"  Starting stream: {cfg['event_hub_namespace']}/{cfg['event_hub_name']} \u2192 {cfg['bronze_table_name']}")
# print(f"  Offsets: {kafka_offsets} | Trigger: {cfg['trigger_interval']} | Max: {cfg['max_events_per_trigger']}")

# raw_stream = (
#     spark.readStream
#     .format("kafka")
#     .options(**kafka_opts)
#     .load()
# )

# bronze_stream = (
#     raw_stream
#     .withColumns({
#         "raw_xml_payload":          F.col("value").cast("string"),
#         "_eventhub_enqueue_ts":     F.col("timestamp"),
#         "_eventhub_offset":         F.col("offset").cast("int"),
#         "_eventhub_partition_id":  F.col("partition").cast("int"),
#         "_source_system_code":      F.lit(cfg.get("source_system_code", cfg["source_system"])),
#         "_division_code":           F.lit(cfg.get("division_code", "")),
#         "_data_domain":             F.lit(cfg.get("data_domain", "")),
#         "_pipeline_name":           F.lit(BATCH_CONTROL_KEY),
#         "_pipeline_run_id":         F.lit(_pipeline_run_id),
#         "_created_ts":              F.current_timestamp(),
#     })
# )

# bronze_stream = (
#     bronze_stream
#     .withColumns({
#         "_record_guid": F.md5(F.concat_ws(
#             "-",
#             F.coalesce(F.col("_eventhub_partition_id").cast("string"), F.lit("")),
#             F.coalesce(F.col("_eventhub_offset").cast("string"), F.lit(""))
#         )),
#     })
#     .select(
#         "_record_guid", "raw_xml_payload",
#         "_eventhub_enqueue_ts", "_eventhub_offset", "_eventhub_partition_id",
#         "_source_system_code", "_division_code", "_data_domain",
#         "_pipeline_name", "_pipeline_run_id", "_created_ts"
#     )
# )

# partition_col = "_created_ts"
# bronze_stream = bronze_stream.withColumn(
#     "partition_date",
#     F.to_date(F.col(partition_col))
# )

# writer = (
#     bronze_stream.writeStream
#     .format("delta")
#     .outputMode("append")
#     .option("checkpointLocation", cfg["checkpoint_location"])
#     .option("mergeSchema", "true")
#     .option("spark.databricks.delta.optimizeWrite.enabled", "true")
#     .trigger(processingTime=cfg["trigger_interval"])
#     .partitionBy("partition_date")
# )

# streaming_query = writer.toTable(cfg["bronze_table_name"])

# print(f"\n\u2713 Streaming query started")
# print(f"  Query ID : {streaming_query.id}")
# print(f"  Target   : {cfg['bronze_table_name']}")
# print(f"  Partition: {partition_col}")